# Acoplamiento por gradiente cruzado

**Nivel 1 — Fundamentos**  
**Prerrequisitos:** nociones de Python; 35 min.

## Objetivos

Siga la evidencia desde el contrato fuente hasta el resultado. Declare unidades y supuestos, reproduzca el cálculo e identifique qué inferencia no respalda la evidencia.

## Preparación

Usa módulos y registros versionados desde la raíz descubierta; no descarga ni modifica fuentes. Instale versiones exactas: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Datos

Ambos modelos 3D son sintéticos y sus recibos guardan coordenadas y métricas por método.

## Método

Densidad gravimétrica en g/cm³, susceptibilidad magnética adimensional y datos gravimétricos en mGal. Un prior de límites compartidos puede expresarse con gradiente cruzado `g=∇ρ×∇κ`; penalizar `||g||²` favorece gradientes paralelos sin igualar amplitudes. Es supuesto estructural, no ley física, y perjudica si los límites difieren. `JOINT_SHARED` es sintético 3D con semilla, 16×24×28 celdas y pasos 80/80/70 m, usando ecuaciones integrales SimPEG 0.25.2 y SciPy. Grafique verdad, inversión independiente y conjunta del corte x-z físico más próximo al y central guardado; use los centros guardados, no un reshape supuesto. Es replay de arreglos, no inversión nueva.

Los operadores tienen unidades, cobertura y espacios nulos diferentes. El término cruzado depende de discretización y escala; gradientes paralelos no implican amplitudes iguales ni geología compartida siempre. Compare predicciones de ambos levantamientos además del error contra verdad. Revise unidades y máscaras antes de interpretar razones pequeñas.

## Resultados

Lea ejes físicos y unidades; distinga ajuste, error contra verdad sintética y acuerdo con etiquetas humanas. El recálculo debe concordar con el recibo. Un modelo graficado no demuestra unicidad, transferencia a campo ni resolución.

### Diagnostic 1

Inspeccione el recibo y procedencia del caso.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Load both seeded synthetic 3D artifacts and verify grid units/provenance.
import json
from pathlib import Path
cases={n:json.loads((root/f"data/derived/v2/{n}/regularization.json").read_text(encoding="utf-8")) for n in ("JOINT_SHARED","JOINT_CONFLICT")}
for n,d in cases.items(): print(n,"synthetic",d["provenance"]["synthetic"],"shape",d["grid"]["shape"],"spacing m",d["grid"]["spacing"],"density",d["units"],"engine",d["engine"])

### Diagnostic 2

Calcule la comparación y grafíquela con unidades físicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot truth and saved independent/joint density on the physical x-z slice at median y.
import matplotlib.pyplot as plt
d=cases["JOINT_SHARED"]; c=np.asarray(d["grid"]["centers"],float); y=np.median(np.unique(c[:,1])); s=np.isclose(c[:,1],y); vals=[np.asarray(d["truth"]),np.asarray(d["methods"]["joint-uncoupled"]["model"]),np.asarray(d["methods"]["joint"]["model"])]
fig,ax=plt.subplots(1,3,figsize=(11,3.5),constrained_layout=True)
for a,v,t in zip(ax,vals,["truth","independent","coupled"]): im=a.scatter(c[s,0],-c[s,2],c=v[s],s=28,cmap="viridis"); a.set(title=t,xlabel="Easting (m)",ylabel="Depth below datum (m)",aspect="equal"); fig.colorbar(im,ax=a,label="Density (g/cm³)")
plt.show(); print("slice northing m",y,"cells",s.sum())

## Resumen

La conclusión depende de geometría, adquisición, ruido, partición, parametrización y derechos de fuente. Conserve negativos y nombre la observación o control siguiente.

## Referencias

- [https://docs.simpeg.xyz/latest/content/user-guide/tutorials/13_joint_inversion/plot_inv_3_cross_gradient_pf.html](https://docs.simpeg.xyz/latest/content/user-guide/tutorials/13_joint_inversion/plot_inv_3_cross_gradient_pf.html)
- [https://www.pygimli.org/user-guide/inversion/](https://www.pygimli.org/user-guide/inversion/)